# v7.8 — Ensemble fuerte: BETO + CNN chars + MLP + clásicos + votación

**No re-entrena BETO** (usa el checkpoint v7.2 que da ~0.275 confiable).

**6 modelos diversos:**
1. **BETO** — inferencia desde checkpoint v7.2. Transformer preentrenado.
2. **CNN de caracteres** — deep learning desde cero, a nivel carácter. Independiente de BETO, buena para OCR (capta 's larga' etc directamente).
3. **MLP** — sobre [embeddings BETO 768d + features hand-crafted 27d]. No-linealidad sobre densos.
4. **TF-IDF + LogReg** — clásico lineal sobre char n-grams.
5. **LinearSVC calibrado** — SVM lineal, otra frontera de decisión.
6. **Regresión ordinal** — predice la década como número real y redondea. Aprovecha el orden.

**Combinación:** se comparan soft voting, hard voting y weighted soft voting; gana el de mejor val acc.

Tiempo estimado: ~1.5h en T4 (la CNN es lo más lento; BETO ya entrenado).

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import re, gc, math, random, unicodedata, itertools
from datetime import datetime
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
from scipy.special import softmax

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

ROOT = Path('/kaggle/working')
DATA_DIR = Path('/kaggle/input/datasets/dboliv/data-kaggle')                # AJUSTA
BETO_V72_CKPT = '/kaggle/input/datasets/dboliv/ckpt-beto/checkpoint-5904'   # AJUSTA
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

BETO_NAME = 'dccuchile/bert-base-spanish-wwm-cased'
MAX_LENGTH = 384
EVAL_STRIDE = 128
BATCH_SIZE = 32

print('device:', DEVICE, '| GPUs visibles:', torch.cuda.device_count())


device: cuda | GPUs visibles: 1


In [2]:
# ============ Carga + preprocesamiento ============
train_raw = pd.read_csv(DATA_DIR / 'train.csv', engine='python', quotechar='"', on_bad_lines='warn')
eval_raw  = pd.read_csv(DATA_DIR / 'eval.csv',  engine='python', quotechar='"', on_bad_lines='warn')

def normalize_text(text):
    text = '' if pd.isna(text) else str(text)
    text = unicodedata.normalize('NFC', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C' or ch in '\n\t ')
    return re.sub(r'\s+', ' ', text).strip()

train = train_raw.copy()
eval_df = eval_raw.copy()
train['text'] = train['text'].map(normalize_text)
eval_df['text'] = eval_df['text'].map(normalize_text)
train = train.dropna(subset=['decade']).copy()
train = train[train['text'].str.len() > 0].copy()
train['decade'] = train['decade'].astype(int)
train = train.drop_duplicates(subset=['text', 'decade']).reset_index(drop=True)
eval_df = eval_df[eval_df['text'].str.len() > 0].copy().reset_index(drop=True)

labels = sorted(train['decade'].unique().tolist())
label2id = {d: i for i, d in enumerate(labels)}
id2label = {i: d for d, i in label2id.items()}
num_labels = len(labels)
train['label'] = train['decade'].map(label2id).astype(int)

vc = train['label'].value_counts()
strat_eligible = train['label'].isin(vc[vc >= 2].index)
pool = train[strat_eligible].reset_index(drop=True)
singletons = train[~strat_eligible].reset_index(drop=True)
train_df, val_df = train_test_split(pool, test_size=0.10, stratify=pool['label'], random_state=SEED)
train_df = pd.concat([train_df, singletons], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

val_true_lbl = val_df['label'].to_numpy()
val_true_dec = val_df['decade'].to_numpy()
print(f'train/val: {len(train_df)}/{len(val_df)}  num_labels: {num_labels}')


def align_proba(proba, classes_, num_labels):
    out = np.zeros((proba.shape[0], num_labels), dtype=np.float32)
    for j, c in enumerate(classes_):
        out[:, int(c)] = proba[:, j]
    return out

def report(name, val_proba):
    pred = np.argmax(val_proba, axis=-1)
    acc = accuracy_score(val_true_lbl, pred)
    mae = mean_absolute_error(val_true_dec, [id2label[i] for i in pred])
    print(f'  [{name:32s}] val acc={acc:.4f}  mae={mae:.4f}')
    return acc

# Soft labels ordinales (compartidas por CNN y MLP)
def make_soft_label_matrix(num_labels, sigma=1.5):
    idx = np.arange(num_labels)
    M = np.zeros((num_labels, num_labels), dtype=np.float32)
    for k in range(num_labels):
        w = np.exp(-((idx - k) ** 2) / (2.0 * sigma ** 2))
        M[k] = w / w.sum()
    return M

SOFT_TARGETS = make_soft_label_matrix(num_labels, 1.5)
SOFT_TARGETS_T = torch.tensor(SOFT_TARGETS, dtype=torch.float32)


train/val: 28231/3137  num_labels: 39


## (1) BETO — inferencia desde checkpoint v7.2 + extracción de embeddings

Una sola carga de BETO sirve para dos cosas: (a) sus logits de clasificación, (b) los embeddings mean-pooled que alimentan el MLP.

In [3]:
def build_chunked_for_inference(df, tokenizer, max_length=MAX_LENGTH, stride=EVAL_STRIDE):
    enc = tokenizer(df['text'].tolist(), truncation=True, max_length=max_length,
                    stride=stride, return_overflowing_tokens=True, padding=False)
    doc_ids = np.array(enc.pop('overflow_to_sample_mapping'), dtype=np.int64)
    ds = HFDataset.from_dict({'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']})
    return ds, doc_ids

def avg_logits_by_doc(chunk_logits, doc_ids, n_docs):
    out = np.zeros((n_docs, chunk_logits.shape[1]), dtype=np.float32)
    counts = np.zeros(n_docs, dtype=np.int32)
    for i, d in enumerate(doc_ids):
        out[int(d)] += chunk_logits[i]; counts[int(d)] += 1
    counts = np.maximum(counts, 1)
    return out / counts[:, None]

def make_inference_args(output_dir):
    return TrainingArguments(
        output_dir=str(output_dir), per_device_eval_batch_size=BATCH_SIZE,
        fp16=torch.cuda.is_available(), report_to='none',
        do_train=False, do_eval=False, save_strategy='no', logging_strategy='no',
    )

print(f'Cargando BETO v7.2: {BETO_V72_CKPT}')
beto_tokenizer = AutoTokenizer.from_pretrained(BETO_NAME, use_fast=True)
beto_clf = AutoModelForSequenceClassification.from_pretrained(BETO_V72_CKPT).to(DEVICE).eval()

beto_trainer = Trainer(model=beto_clf, args=make_inference_args(ROOT / 'tmp_beto'),
                       data_collator=DataCollatorWithPadding(tokenizer=beto_tokenizer))

print('Inferencia BETO logits (val, eval)...')
val_ds, val_doc   = build_chunked_for_inference(val_df,  beto_tokenizer)
eval_ds, eval_doc = build_chunked_for_inference(eval_df, beto_tokenizer)
val_pred  = beto_trainer.predict(val_ds)
eval_pred = beto_trainer.predict(eval_ds)
val_logits_beto  = avg_logits_by_doc(val_pred.predictions,  val_doc,  n_docs=len(val_df))
eval_logits_beto = avg_logits_by_doc(eval_pred.predictions, eval_doc, n_docs=len(eval_df))
val_probs_beto  = softmax(val_logits_beto,  axis=-1)
eval_probs_beto = softmax(eval_logits_beto, axis=-1)
report('BETO (ckpt v7.2)', val_probs_beto)

del beto_trainer, beto_clf, val_ds, eval_ds, val_pred, eval_pred
gc.collect(); torch.cuda.empty_cache()


Cargando BETO v7.2: /kaggle/input/datasets/dboliv/ckpt-beto/checkpoint-5904


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inferencia BETO logits (val, eval)...


  [BETO (ckpt v7.2)                ] val acc=0.2754  mae=2.9691


In [4]:
# Extracción de embeddings mean-pooled de BETO (para el MLP)
@torch.no_grad()
def extract_embeddings(encoder, tokenizer, texts, max_length=MAX_LENGTH, batch_size=BATCH_SIZE):
    encoder.eval()
    hidden = encoder.config.hidden_size
    embs = np.zeros((len(texts), hidden), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        enc = tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(DEVICE)
        out = encoder(**enc)
        last = out.last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1).float()
        pooled = (last * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        embs[i:i+len(chunk)] = pooled.cpu().numpy()
    return embs

print('Extrayendo embeddings de BETO...')
beto_encoder = AutoModel.from_pretrained(BETO_V72_CKPT).to(DEVICE).eval()
emb_train = extract_embeddings(beto_encoder, beto_tokenizer, train_df['text'].tolist())
emb_val   = extract_embeddings(beto_encoder, beto_tokenizer, val_df['text'].tolist())
emb_eval  = extract_embeddings(beto_encoder, beto_tokenizer, eval_df['text'].tolist())
print(f'  embeddings: train {emb_train.shape}  val {emb_val.shape}  eval {emb_eval.shape}')

del beto_encoder, beto_tokenizer
gc.collect(); torch.cuda.empty_cache()


Extrayendo embeddings de BETO...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/datasets/dboliv/ckpt-beto/checkpoint-5904
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  embeddings: train (28231, 768)  val (3137, 768)  eval (3490, 768)


## Features hand-crafted (para concatenar al MLP)

In [5]:
def extract_handcrafted_features(text):
    t = text.lower()
    n = max(len(t), 1)
    words = t.split()
    nw = max(len(words), 1)
    n_f = t.count('f'); n_s = t.count('s'); n_u = t.count('u'); n_v = t.count('v')
    n_i = t.count('i'); n_j = t.count('j'); n_cc = t.count('ç'); n_nt = t.count('ñ')
    f_prevocal   = len(re.findall(r'f[aeiou]', t))
    u_intervocal = len(re.findall(r'[aeiou]u[aeiou]', t))
    v_initial = sum(1 for w in words if w.startswith('v'))
    n_ff = len(re.findall(r'ff', t)); n_ss = len(re.findall(r'ss', t))
    iesse = len(re.findall(r'iesse', t)); asse = len(re.findall(r'asse', t))
    fazer = len(re.findall(r'\bfaz[a-z]+', t))
    auer  = len(re.findall(r'\bauer\b|\bauia\b|\bhuuo\b', t))
    wl = [len(w) for w in words]
    mean_wl = np.mean(wl) if wl else 0.0
    std_wl  = np.std(wl) if wl else 0.0
    short_r = sum(1 for l in wl if l <= 2) / nw
    long_r  = sum(1 for l in wl if l >= 10) / nw
    n_alpha = sum(1 for c in text if c.isalpha())
    n_digit = sum(1 for c in text if c.isdigit())
    n_upper = sum(1 for c in text if c.isupper())
    n_punct = sum(1 for c in text if not c.isalnum() and not c.isspace())
    return np.array([
        n_f/n, n_s/n, n_u/n, n_v/n, n_i/n, n_j/n, n_cc/n, n_nt/n,
        f_prevocal/max(n_f,1), u_intervocal/max(n_u,1), v_initial/nw,
        n_ff/n, n_ss/n, iesse/nw, asse/nw, fazer/nw, auer/nw,
        mean_wl, std_wl, short_r, long_r,
        n_alpha/n, n_digit/n, n_upper/n, n_punct/n,
        np.log1p(len(text)), np.log1p(nw),
    ], dtype=np.float32)

print('Extrayendo features hand-crafted...')
X_hc_train = np.stack([extract_handcrafted_features(t) for t in train_df['text']])
X_hc_val   = np.stack([extract_handcrafted_features(t) for t in val_df['text']])
X_hc_eval  = np.stack([extract_handcrafted_features(t) for t in eval_df['text']])

# Normalizar hand-crafted (z-score con stats de train)
hc_mean = X_hc_train.mean(0, keepdims=True)
hc_std  = X_hc_train.std(0, keepdims=True) + 1e-6
X_hc_train = (X_hc_train - hc_mean) / hc_std
X_hc_val   = (X_hc_val   - hc_mean) / hc_std
X_hc_eval  = (X_hc_eval  - hc_mean) / hc_std
print(f'  hand-crafted: {X_hc_train.shape}')


Extrayendo features hand-crafted...
  hand-crafted: (28231, 27)


## (2) MLP sobre [embeddings BETO + hand-crafted]

Input 768+27=795 dims densas. 2 capas ocultas con BatchNorm y dropout. Entrenado con KL-div sobre soft labels ordinales.

In [6]:
# Normalizar embeddings también
emb_mean = emb_train.mean(0, keepdims=True)
emb_std  = emb_train.std(0, keepdims=True) + 1e-6
emb_train_n = (emb_train - emb_mean) / emb_std
emb_val_n   = (emb_val   - emb_mean) / emb_std
emb_eval_n  = (emb_eval  - emb_mean) / emb_std

# Concatenar
X_mlp_train = np.hstack([emb_train_n, X_hc_train]).astype(np.float32)
X_mlp_val   = np.hstack([emb_val_n,   X_hc_val]).astype(np.float32)
X_mlp_eval  = np.hstack([emb_eval_n,  X_hc_eval]).astype(np.float32)
print(f'MLP input dims: {X_mlp_train.shape[1]}')


class MLP(nn.Module):
    def __init__(self, in_dim, num_labels, h1=512, h2=256, p=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(p),
            nn.Linear(h1, h2),     nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(p * 0.75),
            nn.Linear(h2, num_labels),
        )
    def forward(self, x):
        return self.net(x)


def train_torch_classifier(model, X_tr, y_tr, X_va, y_va, epochs=60, lr=1e-3, bs=256, patience=8):
    """Entrena un clasificador torch con KL-div sobre soft labels. Devuelve mejor estado."""
    model = model.to(DEVICE)
    Xtr = torch.tensor(X_tr); ytr = torch.tensor(y_tr, dtype=torch.long)
    Xva = torch.tensor(X_va).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    soft = SOFT_TARGETS_T.to(DEVICE)
    n = len(Xtr)
    best_acc, best_state, no_improve = -1, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            xb = Xtr[idx].to(DEVICE); yb = ytr[idx].to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = F.kl_div(F.log_softmax(logits, -1), soft[yb], reduction='batchmean')
            loss.backward(); opt.step()
        sched.step()
        # Eval
        model.eval()
        with torch.no_grad():
            va_logits = model(Xva).cpu().numpy()
        va_acc = accuracy_score(y_va, np.argmax(va_logits, -1))
        if va_acc > best_acc:
            best_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_acc


@torch.no_grad()
def predict_torch(model, X):
    model.eval()
    out = model(torch.tensor(X).to(DEVICE)).cpu().numpy()
    return softmax(out, axis=-1)


print('Entrenando MLP...')
mlp = MLP(X_mlp_train.shape[1], num_labels)
mlp, mlp_best_acc = train_torch_classifier(
    mlp, X_mlp_train, train_df['label'].to_numpy(), X_mlp_val, val_true_lbl)
val_proba_mlp  = predict_torch(mlp, X_mlp_val)
eval_proba_mlp = predict_torch(mlp, X_mlp_eval)
report('MLP (emb+handcrafted)', val_proba_mlp)


MLP input dims: 795
Entrenando MLP...
  [MLP (emb+handcrafted)           ] val acc=0.2455  mae=2.8747


0.24545744341727765

## (3) CNN de caracteres entrenada desde cero

Independiente de BETO. Vocabulario de caracteres del train. Convoluciones 1D de distintos tamaños de kernel capturan n-gramas de caracteres (incluyendo patrones OCR como la 's larga').

In [7]:
# Vocabulario de caracteres
CHAR_MAXLEN = 600   # caracteres por texto (cubre la mayoría; los largos se truncan)
all_chars = Counter()
for t in train_df['text']:
    all_chars.update(t.lower())
# Mantener caracteres que aparecen >= 5 veces
vocab_chars = ['<pad>', '<unk>'] + [c for c, cnt in all_chars.most_common() if cnt >= 5]
char2idx = {c: i for i, c in enumerate(vocab_chars)}
VOCAB_SIZE = len(vocab_chars)
print(f'vocab de caracteres: {VOCAB_SIZE}')

def encode_chars(text, maxlen=CHAR_MAXLEN):
    t = text.lower()[:maxlen]
    ids = [char2idx.get(c, 1) for c in t]   # 1 = <unk>
    if len(ids) < maxlen:
        ids = ids + [0] * (maxlen - len(ids))  # 0 = <pad>
    return np.array(ids, dtype=np.int64)

X_cnn_train = np.stack([encode_chars(t) for t in train_df['text']])
X_cnn_val   = np.stack([encode_chars(t) for t in val_df['text']])
X_cnn_eval  = np.stack([encode_chars(t) for t in eval_df['text']])
print(f'  char-encoded: {X_cnn_train.shape}')


class CharCNN(nn.Module):
    def __init__(self, vocab_size, num_labels, emb_dim=64, n_filters=128,
                 kernel_sizes=(3, 4, 5, 6), p=0.4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, n_filters, k, padding=k // 2) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(p)
        self.fc = nn.Sequential(
            nn.Linear(n_filters * len(kernel_sizes), 256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, num_labels),
        )
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)              # (B, emb_dim, L)
        feats = []
        for conv in self.convs:
            c = F.relu(conv(e))                       # (B, n_filters, L)
            c = F.max_pool1d(c, c.size(2)).squeeze(2) # (B, n_filters)
            feats.append(c)
        h = torch.cat(feats, dim=1)
        h = self.dropout(h)
        return self.fc(h)


def train_cnn(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=1e-3, bs=128, patience=6):
    model = model.to(DEVICE)
    Xtr = torch.tensor(X_tr); ytr = torch.tensor(y_tr, dtype=torch.long)
    Xva = torch.tensor(X_va).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    soft = SOFT_TARGETS_T.to(DEVICE)
    n = len(Xtr)
    best_acc, best_state, no_improve = -1, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            xb = Xtr[idx].to(DEVICE); yb = ytr[idx].to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = F.kl_div(F.log_softmax(logits, -1), soft[yb], reduction='batchmean')
            loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            # Eval por batches (val puede ser grande)
            preds = []
            for i in range(0, len(Xva), 256):
                preds.append(model(Xva[i:i+256]).cpu().numpy())
            va_logits = np.vstack(preds)
        va_acc = accuracy_score(y_va, np.argmax(va_logits, -1))
        if va_acc > best_acc:
            best_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
        if ep % 5 == 0:
            print(f'  epoch {ep}: val_acc={va_acc:.4f} (best={best_acc:.4f})')
    model.load_state_dict(best_state)
    return model, best_acc

@torch.no_grad()
def predict_cnn(model, X):
    model.eval()
    preds = []
    Xt = torch.tensor(X).to(DEVICE)
    for i in range(0, len(Xt), 256):
        preds.append(model(Xt[i:i+256]).cpu().numpy())
    return softmax(np.vstack(preds), axis=-1)


print('Entrenando CharCNN...')
cnn = CharCNN(VOCAB_SIZE, num_labels)
cnn, cnn_best_acc = train_cnn(cnn, X_cnn_train, train_df['label'].to_numpy(), X_cnn_val, val_true_lbl)
val_proba_cnn  = predict_cnn(cnn, X_cnn_val)
eval_proba_cnn = predict_cnn(cnn, X_cnn_eval)
report('CharCNN (desde cero)', val_proba_cnn)

del cnn
gc.collect(); torch.cuda.empty_cache()


vocab de caracteres: 135
  char-encoded: (28231, 600)
Entrenando CharCNN...
  epoch 0: val_acc=0.0536 (best=0.0536)
  epoch 5: val_acc=0.1208 (best=0.1208)
  epoch 10: val_acc=0.1310 (best=0.1339)
  epoch 15: val_acc=0.1428 (best=0.1450)
  epoch 20: val_acc=0.1546 (best=0.1546)
  epoch 25: val_acc=0.1505 (best=0.1578)
  [CharCNN (desde cero)            ] val acc=0.1578  mae=3.4680


## (4-6) Modelos clásicos sobre TF-IDF

In [8]:
print('Vectorizando TF-IDF char_wb 2-5...')
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2,
                             max_features=250_000, sublinear_tf=True)
X_tf_train = vectorizer.fit_transform(train_df['text'])
X_tf_val   = vectorizer.transform(val_df['text'])
X_tf_eval  = vectorizer.transform(eval_df['text'])
y_train = train_df['label'].to_numpy()

# (4) LogReg
print('LogReg...')
logreg = LogisticRegression(max_iter=2000, n_jobs=-1, C=1.0, class_weight='balanced')
logreg.fit(X_tf_train, y_train)
val_proba_lr  = align_proba(logreg.predict_proba(X_tf_val),  logreg.classes_, num_labels)
eval_proba_lr = align_proba(logreg.predict_proba(X_tf_eval), logreg.classes_, num_labels)
report('TF-IDF + LogReg', val_proba_lr)

# (5) LinearSVC calibrado
print('LinearSVC calibrado (~5 min)...')
svc = CalibratedClassifierCV(LinearSVC(C=0.5, class_weight='balanced', max_iter=3000, random_state=SEED),
                             method='sigmoid', cv=3)
svc.fit(X_tf_train, y_train)
val_proba_svc  = align_proba(svc.predict_proba(X_tf_val),  svc.classes_, num_labels)
eval_proba_svc = align_proba(svc.predict_proba(X_tf_eval), svc.classes_, num_labels)
report('LinearSVC (calibrado)', val_proba_svc)

# (6) Regresión ordinal: Ridge sobre la década numérica
print('Regresión ordinal (Ridge sobre década)...')
y_train_dec = train_df['decade'].to_numpy().astype(np.float32)
ridge = Ridge(alpha=1.0)
ridge.fit(X_tf_train, y_train_dec)

def reg_to_proba(reg_pred, labels, sigma=1.0):
    """Convierte una predicción numérica de década a una distribución sobre clases,
       gaussiana centrada en la década predicha."""
    labels_arr = np.array(labels, dtype=np.float32)
    probs = np.zeros((len(reg_pred), len(labels)), dtype=np.float32)
    for i, p in enumerate(reg_pred):
        w = np.exp(-((labels_arr - p) ** 2) / (2.0 * sigma ** 2))
        probs[i] = w / w.sum()
    return probs

val_reg_pred  = ridge.predict(X_tf_val)
eval_reg_pred = ridge.predict(X_tf_eval)
val_proba_reg  = reg_to_proba(val_reg_pred,  labels, sigma=1.0)
eval_proba_reg = reg_to_proba(eval_reg_pred, labels, sigma=1.0)
report('Regresión ordinal (Ridge)', val_proba_reg)


Vectorizando TF-IDF char_wb 2-5...
LogReg...
  [TF-IDF + LogReg                 ] val acc=0.2700  mae=4.0398
LinearSVC calibrado (~5 min)...
  [LinearSVC (calibrado)           ] val acc=0.2767  mae=3.7689
Regresión ordinal (Ridge sobre década)...
  [Regresión ordinal (Ridge)       ] val acc=0.0736  mae=4.5846


0.0736372330251833

## Resumen de los 6 modelos

In [9]:
all_models = {
    'BETO':           (val_probs_beto, eval_probs_beto),
    'CharCNN':        (val_proba_cnn,  eval_proba_cnn),
    'MLP':            (val_proba_mlp,  eval_proba_mlp),
    'TF-IDF LogReg':  (val_proba_lr,   eval_proba_lr),
    'LinearSVC':      (val_proba_svc,  eval_proba_svc),
    'Reg. ordinal':   (val_proba_reg,  eval_proba_reg),
}

print('=== Individuales en val ===')
indiv_acc = {}
for name, (vp, _) in all_models.items():
    indiv_acc[name] = report(name, vp)


=== Individuales en val ===
  [BETO                            ] val acc=0.2754  mae=2.9691
  [CharCNN                         ] val acc=0.1578  mae=3.4680
  [MLP                             ] val acc=0.2455  mae=2.8747
  [TF-IDF LogReg                   ] val acc=0.2700  mae=4.0398
  [LinearSVC                       ] val acc=0.2767  mae=3.7689
  [Reg. ordinal                    ] val acc=0.0736  mae=4.5846


## Votación: soft / hard / weighted

In [10]:
model_names = list(all_models.keys())
val_probs_all  = [all_models[n][0] for n in model_names]
eval_probs_all = [all_models[n][1] for n in model_names]

# Soft voting uniforme
soft_val  = np.mean(val_probs_all,  axis=0)
soft_eval = np.mean(eval_probs_all, axis=0)
acc_soft = report('SOFT voting (uniforme)', soft_val)

# Hard voting (BETO desempata; es índice 0)
def hard_vote(probs_list, tiebreak_idx=0):
    preds = np.stack([np.argmax(p, axis=-1) for p in probs_list], axis=1)
    out = np.zeros(preds.shape[0], dtype=int)
    for i in range(preds.shape[0]):
        votes = Counter(preds[i])
        top = votes.most_common()
        winners = [c for c, cnt in top if cnt == top[0][1]]
        out[i] = winners[0] if len(winners) == 1 else preds[i, tiebreak_idx]
    return out

hard_pred = hard_vote(val_probs_all, tiebreak_idx=0)
acc_hard = accuracy_score(val_true_lbl, hard_pred)
mae_hard = mean_absolute_error(val_true_dec, [id2label[i] for i in hard_pred])
print(f'  [{"HARD voting":32s}] val acc={acc_hard:.4f}  mae={mae_hard:.4f}')

# Weighted soft voting (grid)
print('\nGrid de pesos (weighted soft voting)...')
best = None
step_options = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
for w in itertools.product(step_options, repeat=len(model_names)):
    if abs(sum(w) - 1.0) > 0.01:
        continue
    probs = sum(wi * vi for wi, vi in zip(w, val_probs_all))
    acc = accuracy_score(val_true_lbl, np.argmax(probs, axis=-1))
    if best is None or acc > best[1]:
        best = (w, acc)
w_best, acc_weighted = best
print(f'  mejor: {dict(zip(model_names, [round(x,2) for x in w_best]))}')
print(f'  [{"WEIGHTED soft voting":32s}] val acc={acc_weighted:.4f}')


  [SOFT voting (uniforme)          ] val acc=0.2869  mae=2.7928
  [HARD voting                     ] val acc=0.2869  mae=2.9640

Grid de pesos (weighted soft voting)...
  mejor: {'BETO': 0.1, 'CharCNN': 0.3, 'MLP': 0.1, 'TF-IDF LogReg': 0.1, 'LinearSVC': 0.4, 'Reg. ordinal': 0.0}
  [WEIGHTED soft voting            ] val acc=0.3137


## Elegir mejor esquema y submission

In [11]:
schemes = {'soft_uniform': acc_soft, 'hard_voting': acc_hard, 'weighted_soft': acc_weighted}
print('=== Esquemas (val acc) ===')
for name, acc in sorted(schemes.items(), key=lambda x: -x[1]):
    print(f'  {name:18s}: {acc:.4f}')

best_scheme = max(schemes, key=schemes.get)
best_indiv  = max(indiv_acc, key=indiv_acc.get)
print(f'\nMejor esquema:    {best_scheme}  ({schemes[best_scheme]:.4f})')
print(f'Mejor individual: {best_indiv}  ({indiv_acc[best_indiv]:.4f})')

# Generar eval del mejor esquema
if best_scheme == 'soft_uniform':
    eval_pred_ids = np.argmax(soft_eval, axis=-1)
elif best_scheme == 'hard_voting':
    eval_pred_ids = hard_vote(eval_probs_all, tiebreak_idx=0)
else:
    eval_final = sum(wi * ei for wi, ei in zip(w_best, eval_probs_all))
    eval_pred_ids = np.argmax(eval_final, axis=-1)

answers = [id2label[int(i)] for i in eval_pred_ids]
submission = pd.DataFrame({'id': eval_df['id'].to_numpy(), 'answer': answers})
ts = datetime.now().strftime('%Y%m%d_%H%M')
out_name = ROOT / f'submission_v78_{best_scheme}_{ts}.csv'
submission.to_csv(out_name, index=False)
submission.to_csv(ROOT / 'submission.csv', index=False)
print(f'\nsaved {out_name} {submission.shape}')
print(submission.head())

# Guardar todas las predicciones para iterar sin re-correr
np.savez(ROOT / 'predictions_v78.npz',
    **{f'val_{n}': all_models[n][0] for n in model_names},
    **{f'eval_{n}': all_models[n][1] for n in model_names},
    val_true_lbl=val_true_lbl, val_true_dec=val_true_dec,
)
print('predicciones guardadas en predictions_v78.npz')

if schemes[best_scheme] <= indiv_acc[best_indiv] + 0.003:
    print('\n⚠️  La votación no supera claramente al mejor modelo individual (dentro del ruido).')
else:
    print(f'\n✓ La votación mejora sobre el mejor individual por {schemes[best_scheme]-indiv_acc[best_indiv]:.4f}.')


=== Esquemas (val acc) ===
  weighted_soft     : 0.3137
  soft_uniform      : 0.2869
  hard_voting       : 0.2869

Mejor esquema:    weighted_soft  (0.3137)
Mejor individual: LinearSVC  (0.2767)

saved /kaggle/working/submission_v78_weighted_soft_20260520_2229.csv (3490, 2)
   id  answer
0   0     173
1   1     185
2   2     150
3   3     172
4   4     153
predicciones guardadas en predictions_v78.npz

✓ La votación mejora sobre el mejor individual por 0.0370.
